# Descarga del diccionario de datos — Módulos F1, F9 y F51
## ECV 2025 · DANE · Encuesta Nacional de Calidad de Vida

---

### ¿Qué hace este notebook?

Descarga los metadatos de las variables de tres módulos de la ECV 2025 del DANE y los consolida en un único CSV:

| Módulo | Nombre | Variables | Casos |
|--------|--------|-----------|-------|
| F1 | Datos de la vivienda | 46 | 86.848 |
| F9 | Características y composición del hogar | 78 | 235.350 |
| F51 | Condiciones de vida del hogar y tenencia de bienes | 143 | 87.060 |

### ¿Por qué se usa el endpoint JSON y no scraping HTML?

El portal NADA del DANE expone un endpoint JSON por variable:
```
https://microdatos.dane.gov.co/index.php/metadata/export_variable/905/{variable_id}/json
```
Ese endpoint devuelve en una sola respuesta todos los metadatos: pregunta literal, tipo, rango, universo y categorías con frecuencias. Es más limpio y robusto que parsear HTML.

### ¿Por qué los IDs de variables se declaran explícitamente en F1?

Los módulos F9 y F51 tienen IDs de variables consecutivos, así que basta con un rango numérico. El módulo F1, en cambio, tiene IDs **no consecutivos**: la mayoría están entre V1 y V45, pero tres variables tienen IDs V2819, V2935 y V2936, que son saltos grandes producto de cómo el sistema asignó los IDs históricamente. Si se iterara el rango completo V1–V2936 se harían ~2.900 peticiones innecesarias. Por eso F1 usa una lista explícita de IDs extraídos directamente del diccionario del portal.

### Estructura del CSV resultante

| Columna | Descripción |
|---|---|
| `modulo` | Código del módulo: F1, F9 o F51 |
| `modulo_nombre` | Nombre descriptivo del módulo |
| `variable_id` | ID interno del sistema (ej. V4502) |
| `nombre_variable` | Código del campo en el dataset (ej. P1079) |
| `pregunta` | Texto de la etiqueta/pregunta |
| `tipo` | Tipo de dato: numeric / character |
| `intervalo` | discrete o continuous |
| `casos_validos` | Registros con respuesta |
| `casos_invalidos` | Registros sin respuesta |
| `valor_min` | Valor mínimo del rango |
| `valor_max` | Valor máximo del rango |
| `universo` | Población a la que aplica la pregunta |
| `categorias` | Categorías de respuesta con frecuencia, separadas por `;` |

### Requisitos

Solo librería estándar de Python: `urllib`, `json`, `csv`, `time`. La celda de vista previa usa `pandas`, que es opcional.

---
## Celda 1 — Definición de módulos y sus variables

Cada módulo se define como un diccionario con su nombre y la lista de IDs de variables.

- **F9 y F51**: IDs consecutivos → se genera la lista con `range()`.
- **F1**: IDs no consecutivos → lista explícita extraída del portal del DANE.

In [1]:
# ─── PARÁMETROS GLOBALES ──────────────────────────────────────────────────────

SURVEY_ID      = "905"                          # ID de la encuesta ECV 2025
PAUSA_SEGUNDOS = 0.3                            # Pausa entre peticiones HTTP
ARCHIVO_SALIDA = "dane_ecv2025_diccionario.csv" # Nombre del CSV de salida

# ─── DEFINICIÓN DE MÓDULOS ────────────────────────────────────────────────────

MODULOS = [
    {
        "codigo": "F1",
        "nombre": "Datos de la vivienda",
        # F1 tiene IDs no consecutivos: la mayoría en V1-V45,
        # pero tres variables tienen IDs V2819, V2935 y V2936.
        # Se listan explícitamente para evitar ~2900 peticiones innecesarias.
        "ids": [
            "V1", "V2", "V3", "V4", "V5", "V6", "V7", "V8", "V9", "V10",
            "V11", "V2935", "V2936", "V14", "V15", "V16", "V17", "V18",
            "V19", "V20", "V21", "V22", "V23", "V24", "V25", "V26", "V27",
            "V28", "V29", "V30", "V31", "V32", "V33", "V34", "V35", "V36",
            "V37", "V38", "V39", "V40", "V41", "V42", "V43", "V44", "V45",
            "V2819"
        ]
    },
    {
        "codigo": "F9",
        "nombre": "Caracteristicas y composicion del hogar",
        # IDs consecutivos V464-V542, con un salto en V526 (no existe en el módulo).
        "ids": [f"V{i}" for i in range(464, 543) if i != 526]
    },
    {
        "codigo": "F51",
        "nombre": "Condiciones de vida del hogar y tenencia de bienes",
        # IDs completamente consecutivos V4497-V4639.
        "ids": [f"V{i}" for i in range(4497, 4640)]
    },
]

# Resumen
total_vars = sum(len(m["ids"]) for m in MODULOS)
for m in MODULOS:
    print(f"{m['codigo']:4} | {m['nombre']:50} | {len(m['ids'])} variables")
print(f"\nTotal de peticiones a realizar: {total_vars}")
print(f"Tiempo estimado: ~{total_vars * PAUSA_SEGUNDOS / 60:.1f} minutos")

F1   | Datos de la vivienda                               | 46 variables
F9   | Caracteristicas y composicion del hogar            | 78 variables
F51  | Condiciones de vida del hogar y tenencia de bienes | 143 variables

Total de peticiones a realizar: 267
Tiempo estimado: ~1.3 minutos


---
## Celda 2 — Función de descarga

Hace una petición al endpoint JSON del DANE y normaliza la respuesta a un diccionario plano.

**Por qué se simula un User-Agent de navegador:** el servidor bloquea con 403 las peticiones que se identifican como scripts automáticos (`python-urllib`). Declarar un User-Agent de Chrome real hace que la petición pase los filtros del servidor sin necesidad de cookies ni sesión autenticada.

In [2]:
import urllib.request
import json

BASE_URL = "https://microdatos.dane.gov.co/index.php/metadata/export_variable/{survey}/{vid}/json"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept": "application/json, text/plain, */*",
    "Referer": "https://microdatos.dane.gov.co/",
}


def descargar_variable(survey_id: str, vid: str, modulo_codigo: str, modulo_nombre: str) -> dict | None:
    """
    Descarga los metadatos de una variable desde el endpoint JSON del DANE.
    Devuelve un dict con los campos normalizados, o None si la petición falla.
    """
    url = BASE_URL.format(survey=survey_id, vid=vid)
    try:
        req = urllib.request.Request(url, headers=HEADERS)
        with urllib.request.urlopen(req, timeout=20) as resp:
            data = json.loads(resp.read().decode("utf-8"))
    except Exception as e:
        print(f"  ERROR {vid}: {e}")
        return None

    # Casos válidos e inválidos
    # var_sumstat es una lista de estadísticos; cada uno tiene un campo 'type'.
    casos_validos = casos_invalidos = ""
    for s in data.get("var_sumstat") or []:
        if s.get("type") == "vald":
            casos_validos = s.get("value", "")
        elif s.get("type") == "invd":
            casos_invalidos = s.get("value", "")

    # Categorías de respuesta comprimidas en una sola cadena
    # Formato: "1=Jefe/a del hogar (78378); 2=Cónyuge (8682)"
    partes = []
    for cat in data.get("var_catgry") or []:
        valor = cat.get("value", "")
        etiq  = cat.get("labl", "")
        freq  = ""
        for st in cat.get("stats") or []:
            if st.get("type") == "freq":
                freq = st.get("value", "")
        partes.append(f"{valor}={etiq} ({freq})")
    categorias_str = "; ".join(partes)

    rango = data.get("var_val_range") or {}
    fmt   = data.get("var_format") or {}

    return {
        "modulo"         : modulo_codigo,
        "modulo_nombre"  : modulo_nombre,
        "variable_id"    : data.get("vid", ""),
        "nombre_variable": data.get("name", ""),
        "pregunta"       : data.get("labl", ""),
        "tipo"           : fmt.get("type", ""),
        "intervalo"      : data.get("var_intrvl", ""),
        "casos_validos"  : casos_validos,
        "casos_invalidos": casos_invalidos,
        "valor_min"      : rango.get("min", ""),
        "valor_max"      : rango.get("max", ""),
        "universo"       : (data.get("var_universe") or "").replace("\n", " ").strip(),
        "categorias"     : categorias_str,
    }


print("Función lista.")

Función lista.


---
## Celda 3 — Descarga de todos los módulos

Itera módulo por módulo, variable por variable. Imprime el progreso en tiempo real y acumula errores por módulo para revisarlos al final sin interrumpir la descarga.

In [3]:
import time

filas   = []   # filas exitosas (todos los módulos)
errores = {}   # {modulo_codigo: [ids_fallidos]}

for modulo in MODULOS:
    codigo = modulo["codigo"]
    nombre = modulo["nombre"]
    ids    = modulo["ids"]
    errores[codigo] = []

    print(f"\n{'='*60}")
    print(f"Módulo {codigo}: {nombre}")
    print(f"Variables: {len(ids)}")
    print(f"{'='*60}")

    for idx, vid in enumerate(ids, start=1):
        fila = descargar_variable(SURVEY_ID, vid, codigo, nombre)
        if fila:
            filas.append(fila)
            print(f"  [{idx:3}/{len(ids)}] OK   {vid} → {fila['nombre_variable']}")
        else:
            errores[codigo].append(vid)
            print(f"  [{idx:3}/{len(ids)}] FAIL {vid}")
        time.sleep(PAUSA_SEGUNDOS)

print(f"\n{'='*60}")
print("RESUMEN FINAL")
print(f"{'='*60}")
print(f"Total descargadas : {len(filas)}")
for codigo, ids_err in errores.items():
    estado = f"{len(ids_err)} errores" if ids_err else "sin errores"
    print(f"  {codigo}: {estado}")
    if ids_err:
        print(f"    IDs fallidos: {ids_err}")


Módulo F1: Datos de la vivienda
Variables: 46
  [  1/46] OK   V1 → DIRECTORIO
  [  2/46] OK   V2 → SECUENCIA_ENCUESTA
  [  3/46] OK   V3 → SECUENCIA_P
  [  4/46] OK   V4 → ORDEN
  [  5/46] OK   V5 → P1_DEPARTAMENTO
  [  6/46] OK   V6 → P1_MUNICIPIO
  [  7/46] OK   V7 → REGION
  [  8/46] OK   V8 → FEX_C
  [  9/46] OK   V9 → CANT_HOG_COMPLETOS
  [ 10/46] OK   V10 → CANT_HOGARES_VIVIENDA
  [ 11/46] OK   V11 → CLASE
  [ 12/46] OK   V2935 → P2102
  [ 13/46] OK   V2936 → P3155
  [ 14/46] OK   V14 → P3156
  [ 15/46] OK   V15 → P1070
  [ 16/46] OK   V16 → P4005
  [ 17/46] OK   V17 → P4015
  [ 18/46] OK   V18 → P4567
  [ 19/46] OK   V19 → P8520
  [ 20/46] OK   V20 → P8520S1
  [ 21/46] OK   V21 → P8520S1A1
  [ 22/46] OK   V22 → P8520S5
  [ 23/46] OK   V23 → P8520S3
  [ 24/46] OK   V24 → P8520S4
  [ 25/46] OK   V25 → P8520S4A1
  [ 26/46] OK   V26 → P4065
  [ 27/46] OK   V27 → P4065S1
  [ 28/46] OK   V28 → P4065S2
  [ 29/46] OK   V29 → P4065S3
  [ 30/46] OK   V30 → P4065S4
  [ 31/46] OK   V31 → P

---
## Celda 4 — Guardado del CSV

Escribe con encoding `utf-8-sig` (UTF-8 con BOM) para que Excel en Windows detecte el encoding automáticamente y muestre bien tildes, ñ y caracteres especiales del español.

In [4]:
import csv

CAMPOS = [
    "modulo", "modulo_nombre", "variable_id", "nombre_variable",
    "pregunta", "tipo", "intervalo", "casos_validos", "casos_invalidos",
    "valor_min", "valor_max", "universo", "categorias"
]

with open(ARCHIVO_SALIDA, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=CAMPOS)
    writer.writeheader()
    writer.writerows(filas)

print(f"CSV guardado: {ARCHIVO_SALIDA}")
print(f"Filas escritas: {len(filas)} (+ 1 encabezado)")

CSV guardado: dane_ecv2025_diccionario.csv
Filas escritas: 267 (+ 1 encabezado)


---
## Celda 5 — Vista previa del resultado

Muestra las primeras filas de cada módulo para verificar que la descarga fue correcta. Requiere `pandas`.

In [5]:
import pandas as pd

df = pd.read_csv(ARCHIVO_SALIDA, encoding="utf-8-sig")

print(f"Dimensiones totales: {df.shape[0]} filas × {df.shape[1]} columnas\n")
print("Variables por módulo:")
print(df.groupby("modulo")["nombre_variable"].count().to_string())
print()

df.head(10)

Dimensiones totales: 267 filas × 13 columnas

Variables por módulo:
modulo
F1      46
F51    143
F9      78



,modulo,modulo_nombre,variable_id,nombre_variable,pregunta,tipo,intervalo,casos_validos,casos_invalidos,valor_min,valor_max,universo,categorias
0,F1,Datos de la vivienda,V1,DIRECTORIO,DIRECTORIO,numeric,contin,86848,0,NaN,NaN,El universo para la Encuesta de Calidad de Vid...,NaN
1,F1,Datos de la vivienda,V2,SECUENCIA_ENCUESTA,SECUENCIA_ENCUESTA,numeric,discrete,86848,0,NaN,NaN,El universo para la Encuesta de Calidad de Vid...,1=None (86848)
2,F1,Datos de la vivienda,V3,SECUENCIA_P,SECUENCIA_P,numeric,discrete,86848,0,NaN,NaN,El universo para la Encuesta de Calidad de Vid...,1=None (86848)
3,F1,Datos de la vivienda,V4,ORDEN,ORDEN,numeric,discrete,86848,0,NaN,NaN,El universo para la Encuesta de Calidad de Vid...,1=None (86848)
4,F1,Datos de la vivienda,V5,P1_DEPARTAMENTO,Código del departamento,character,discrete,86848,0,NaN,NaN,El universo para la Encuesta de Calidad de Vid...,05=None (3726); 08=None (2621); 11=None (2947)...
5,F1,Datos de la vivienda,V6,P1_MUNICIPIO,Código del municipio,character,discrete,86848,0,NaN,NaN,El universo para la Encuesta de Calidad de Vid...,001=None (28895); 002=None (60); 003=None (102...
6,F1,Datos de la vivienda,V7,REGION,Región,numeric,discrete,86848,0,1.0,9.0,El universo para la Encuesta de Calidad de Vid...,1=None (18277); 2=None (16444); 3=None (16676)...
7,F1,Datos de la vivienda,V8,FEX_C,Factor de expansión,numeric,contin,86848,0,NaN,NaN,El universo para la Encuesta de Calidad de Vid...,NaN
8,F1,Datos de la vivienda,V9,CANT_HOG_COMPLETOS,Cantidad de hogares completos,numeric,discrete,86848,0,NaN,NaN,El universo para la Encuesta de Calidad de Vid...,1=None (86667); 2=None (157); 3=None (20); 4=N...
9,F1,Datos de la vivienda,V10,CANT_HOGARES_VIVIENDA,Cantidad de viviendas en el hogar,numeric,discrete,86848,0,1.0,8.0,El universo para la Encuesta de Calidad de Vid...,1=None (86667); 2=None (157); 3=None (20); 4=N...
